In [ ]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                self.p.v_pd_hf_tweezer_squeeze_power = 3.94
                                 
                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 500.e-6
                
                self.p.amp_imaging = {img_amp}
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 0.1 * np.pi

                self.p.t_tweezer_hold = 15.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 15

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_midpoint)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(ramp_down_painting=True,squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)

                self.raman.pulse(t=self.p.t_continuous_rabi)
                
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [7]:
eBuilder = ExptBuilder()

In [ ]:
img_amp = np.linspace(.2,2.5,50)

freq_raman = np.array([1.19494357e+08, 1.19501388e+08, 1.19508419e+08, 1.19515449e+08,
       1.19522480e+08, 1.19529511e+08, 1.19536542e+08, 1.19543572e+08,
       1.19550603e+08, 1.19557634e+08, 1.19564665e+08, 1.19571695e+08,
       1.19578726e+08, 1.19585757e+08, 1.19592788e+08, 1.19599819e+08,
       1.19606849e+08, 1.19613880e+08, 1.19620911e+08, 1.19627942e+08,
       1.19634972e+08, 1.19642003e+08, 1.19649034e+08, 1.19656065e+08,
       1.19663095e+08, 1.19670126e+08, 1.19677157e+08, 1.19684188e+08,
       1.19691218e+08, 1.19698249e+08, 1.19705280e+08, 1.19712311e+08,
       1.19719341e+08, 1.19726372e+08, 1.19733403e+08, 1.19740434e+08,
       1.19747464e+08, 1.19754495e+08, 1.19761526e+08, 1.19768557e+08,
       1.19775587e+08, 1.19782618e+08, 1.19789649e+08, 1.19796680e+08,
       1.19803710e+08, 1.19810741e+08, 1.19817772e+08, 1.19824803e+08,
       1.19831834e+08, 1.19838864e+08])


for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.2 119495934.0
0  15 values of dummy. 15 total shots. 45 total images expected.
Run ID: 70356
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.22, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.22 pi, x-center = 994, y-center = 820

 Run ID: 70356

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.22, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.22 pi, x-center = 994, y-center = 820

shot 1/15 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.22, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.22 pi, x-center = 994, y-center = 820

shot 2/15 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.22, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.22 pi, x